# 06 · Ingestão Silver — Squad 3 | Henrique Ficht

Lê as tabelas Delta da camada **bronze**, aplica limpeza, cast de tipos e
**regras técnicas de qualidade** da planilha de `Regras de Negócio da Squad 3`,
e salva como **Delta** na camada **silver**.


## Contrato — Regras técnicas aplicadas (10 regras)

Dados inválidos **nunca são descartados** — são sinalizados via colunas `_flag_*`
que a Gold filtra conforme a métrica. Objetivo é preservar a rastreabilidade e permite
re-processamento futuro se a regra mudar.

### `food_fornecedores` (regras T1–T5 da planilha)

| # | Regra | Flag ou tratamento |
|---|---|---|
| T1 | `id_fornecedor` não nulo nem duplicado | `filter(isNotNull)` + dedup por `Window.partitionBy(id_fornecedor).orderBy(bronze_ingested_at.desc)` |
| T2 | `cnpj` com 14 dígitos numéricos | `_flag_cnpj_invalido` (CNPJ de 13 dígitos é auto-corrigido com `lpad(0)`) |
| T3 | `categoria_fornecida` nas raízes de CATEGORIAS ou PERECIVEIS | `_flag_categoria_invalida` com UNION dinâmico das duas fontes |
| T4 | `tipo_fornecedor` na lista de 7 valores permitidos | `_flag_tipo_invalido` |
| T5 | `lead_time_dias > 0` + ranges por tipo (1-3 Produção Própria, 2-20 demais) | `_flag_lead_time_invalido` |

**Validações extras (não exigidas na planilha, mas aplicadas por qualidade):**
- `_flag_uf_invalida` — valida contra as 27 UFs do Brasil
- `_flag_razao_social_vazia` — captura razão social em branco

### `food_lotes_producao` (regras T1–T5 da planilha)

| # | Regra | Flag ou tratamento |
|---|---|---|
| T1 | `id_lote` não nulo nem duplicado | `filter(isNotNull)` + dedup via Window por `bronze_ingested_at.desc` |
| T2 | `sku` deve existir no catálogo (`ecommerce_produtos` ∪ `physical_produtos_pereciveis`) | `_flag_sku_sem_catalogo` |
| T3 | `id_fornecedor` deve existir em `food_fornecedores` (FK) | `_flag_fornecedor_orfao` |
| T4 | `status` em ('Ativo', 'Vencido', 'Recall') | `_flag_status_invalido` |
| T5 | `dt_validade` > `dt_fabricacao` | `_flag_validade_invertida` |

**Validações extras:**
- `_flag_temperatura_invalida` — valida enum ('Refrigerado', 'Ambiente', 'Congelado')
- `_flag_quantidade_invalida` — captura `quantidade_produzida ≤ 0` ou nula

## Pipeline de transformação

Para cada tabela, 6 etapas em ordem:

1. **Deduplicação** por PK (mantém mais recente por `bronze_ingested_at`)
2. **Remoção** das colunas de auditoria da Bronze (`bronze_ingested_at`, `bronze_source_file`, `ano`, `mes`)
3. **Cast explícito** de tipos (não depende de `inferSchema`)
4. **Trim** em todas as strings + normalizações específicas (CNPJ, UF)
5. **Validação com flags** `_flag_*` para cada regra técnica
6. **Auditoria Silver** (`silver_processed_at`) e repartição por `ano`/`mes` da data da própria Silver

## Observações importantes

- **Modo de escrita:** `merge` (upsert por PK, via DataFrame manual — ver BLOCO 5)
- **CNPJ auto-corrigido:** CNPJs com 13 dígitos recebem `lpad(0)` automaticamente
  (comum em import de Excel que suprime zero à esquerda). CNPJs com 11, 12 ou
  8-10 dígitos permanecem como estão, flagados como inválidos.
- **Compatibilidade:** Merge `DeltaTable.merge()`  é implementado via DataFrame
  (union + dedup por PK + overwrite).

In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

## BLOCO 1 — Setup

Carrega credenciais do Service Principal via `.env` e define paths bronze→silver
no ADLS Gen2. Também inclui uma célula de **diagnóstico** que verifica a existência
das Silvers de outras squads (`ecommerce_produtos` da Squad 1 e
`physical_produtos_pereciveis` da Squad 3) — necessárias para a validação de SKU
e categoria.

In [0]:
# ── Configuração de credenciais e paths ────────────────────────────────────
import os
from dotenv import load_dotenv

ENV_PATH = "/Workspace/Users/ik.kukoo@gmail.com/.env"
load_dotenv(ENV_PATH, override=True)

def _require(key):
    value = os.getenv(key)
    if not value:
        raise EnvironmentError(f"Variável '{key}' não encontrada em {ENV_PATH}")
    return value

STORAGE_ACCOUNT = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{_require('ADLS_TENANT_ID')}/oauth2/token",
}

RAW_BASE    = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data"
BRONZE_BASE = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"
SILVER_BASE = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

print("✅ Config carregado.")
print(f"   RAW    → {RAW_BASE}")
print(f"   BRONZE → {BRONZE_BASE}")
print(f"   SILVER → {SILVER_BASE}")

In [0]:
# =======================================================
# Verificar se as Silvers de outras squads já existem
# =======================================================
for tabela in ["ecommerce_produtos", "physical_produtos_pereciveis"]:
    # Tentar nos containers padrão
    paths_possiveis = [
        f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{tabela}",
        f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{tabela}",
    ]
    for path in paths_possiveis:
        try:
            df = spark.read.format("delta").options(**adls_options).load(path)
            print(f"✅ Encontrado: {path} ({df.count()} registros)")
        except Exception as e:
            if "Path does not exist" in str(e) or "PATH_NOT_FOUND" in str(e):
                print(f"❌ Não existe: {path}")
            else:
                print(f"⚠️  Erro: {path} → {str(e)[:100]}")

## BLOCO 2 — Leitura das Bronze + Catálogos de Referência

Lê as duas tabelas Bronze (Delta) e os dois catálogos de SKU que serão usados
para validar `food_lotes_producao.sku` (regra T2 lotes).

A união `ecommerce_produtos ∪ physical_produtos_pereciveis` forma o universo
oficial de SKUs válidos: perecíveis (lojas físicas) + não-perecíveis (e-commerce).

In [0]:
# ── Imports ───────────────────────────────────────────────────────────────
from pyspark.sql.functions import (
    col, current_timestamp, year, month, trim, upper, length,
    regexp_replace, when, lit, row_number, concat_ws,
    split, transform
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType, StringType

# ── Leitura das tabelas Bronze (Delta) ────────────────────────────────────
df_bronze_fornecedores = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(f"{BRONZE_BASE}/food_fornecedores")
)

df_bronze_lotes = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(f"{BRONZE_BASE}/food_lotes_producao")
)

print(f"✅ Bronze food_fornecedores:   {df_bronze_fornecedores.count()} registros | {len(df_bronze_fornecedores.columns)} colunas")
print(f"✅ Bronze food_lotes_producao: {df_bronze_lotes.count()} registros | {len(df_bronze_lotes.columns)} colunas")

In [0]:
# ── Carregar catálogos de SKU das camadas Silver ──────────────────────────
# ecommerce_categorias:          Silver da Squad 1
# physical_produtos_pereciveis:  Silver da Squad 3

skus_ecommerce = (
    spark.read.format("delta").options(**adls_options)
    .load(f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ecommerce_produtos")
    .select(trim(col("sku")).alias("sku_catalogo"))
)
print(f"✅ ecommerce_produtos (Squad 1 Silver): {skus_ecommerce.count()} SKUs")

skus_pereciveis = (
    spark.read.format("delta").options(**adls_options)
    .load(f"{SILVER_BASE}/physical_produtos_pereciveis")
    .select(trim(col("sku")).alias("sku_catalogo"))
)
print(f"✅ physical_produtos_pereciveis (Squad 3 Silver): {skus_pereciveis.count()} SKUs")

# Unificar
skus_validos = skus_ecommerce .union(skus_pereciveis).distinct()
print(f"\n✅ Catálogo unificado: {skus_validos.count()} SKUs válidos")

## BLOCO 3 — Transformação Silver: `food_fornecedores`

Aplica as 6 etapas de transformação. Produz o DataFrame `df_silver_fornecedores`
com todas as flags `_flag_*` populadas e colunas de partição (`ano`, `mes` baseadas
em `dt_cadastro`).

**Observação sobre a regra T5 (lead time):** a validação de ranges depende do
`tipo_fornecedor`. Se `tipo_fornecedor` já for inválido (flagado por T4), a flag
T5 aplica o range padrão (2-20 dias) — ambas as flags podem ficar `True`
simultaneamente para o mesmo registro. É esperado, não é bug.

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# SILVER — food_fornecedores
# Etapas: Dedup → Remover colunas Bronze → Cast → Trim/Normalização →
#         Validação (flags) → Auditoria
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🔧 Silver: food_fornecedores")
print("=" * 60)

from pyspark.sql.functions import col, trim, upper, split, concat_ws, transform, when, lit, current_timestamp, year, month, regexp_replace, lpad, length
df = df_bronze_fornecedores

# ── ETAPA 1: DEDUPLICAÇÃO (Regra 1 — PK) ────────────────────────────────
antes = df.count()
df = df.filter(col("id_fornecedor").isNotNull())
w = Window.partitionBy("id_fornecedor").orderBy(col("bronze_ingested_at").desc())
df = df.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")
depois = df.count()
print(f"   Etapa 1 — Dedup PK: {antes} → {depois} (removidos {antes - depois})")

# ── ETAPA 2: REMOVER COLUNAS DE AUDITORIA DA BRONZE ─────────────────────
df = df.drop("bronze_ingested_at", "bronze_source_file", "ano", "mes")
print("   Etapa 2 — Colunas Bronze removidas")

# ── ETAPA 3: CAST EXPLÍCITO DE TIPOS ────────────────────────────────────
df = (
    df
    .withColumn("id_fornecedor", col("id_fornecedor").cast(IntegerType()))
    .withColumn("cnpj", col("cnpj").cast(StringType()))
    .withColumn("lead_time_dias", col("lead_time_dias").cast(IntegerType()))
    .withColumn("is_ativo", col("is_ativo").cast(BooleanType()))
    .withColumn("dt_cadastro", col("dt_cadastro").cast(TimestampType()))
)
print("   Etapa 3 — Cast de tipos aplicado")

# ── ETAPA 4: TRIM, NORMALIZAÇÃO E FLAG DO CNPJ ─────────────────────────

df = (
    df
    .withColumn("razao_social", trim(col("razao_social")))
    .withColumn("categoria_fornecida", trim(col("categoria_fornecida")))
    .withColumn("tipo_fornecedor", trim(col("tipo_fornecedor")))
    .withColumn("uf_origem", trim(upper(col("uf_origem"))))
)

# CNPJ: remover pontuação, manter só dígitos
df = df.withColumn("cnpj", regexp_replace(col("cnpj"), "[^0-9]", ""))

# Flag ANTES do tratamento — captura CNPJ truncado (chaos proposital)
df = df.withColumn(
    "_flag_cnpj_invalido",
    when((col("cnpj").isNull()) | (length(col("cnpj")) != 14), lit(True))
    .otherwise(lit(False))
)

# Tratar com segurança: lpad SOMENTE para 13 dígitos (zero à esquerda faltante)
# Demais casos (11, 12, 8-10 dígitos) permanecem como estão
df = df.withColumn(
    "cnpj",
    when(length(col("cnpj")) == 13, lpad(col("cnpj"), 14, "0"))
    .otherwise(col("cnpj"))
)

# Certificações: trim de cada item separado por ;
df = df.withColumn(
    "certificacoes",
    concat_ws(";", transform(split(col("certificacoes"), ";"), lambda x: trim(x)))
)
print("   Etapa 4 — Trim, normalização e flag CNPJ aplicados")

# ── ETAPA 5: VALIDAÇÃO — FLAGS DE QUALIDADE ─────────────────────────────

# Regra 3: categoria_fornecida válida — lida das Silvers das tabelas de referência

# Squad 1 — ecommerce_categorias (Silver): somente categorias raiz
categorias_ecommerce = (
    spark.read.format("delta").options(**adls_options)
    .load(f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ecommerce_categorias")
    .filter(col("tipo_categoria") == "Categoria")
    .select(col("nome_categoria").alias("categoria"))
)
print(f"   Categorias ecommerce (Squad 1 Silver): {categorias_ecommerce.count()}")

# Squad 3 — physical_produtos_pereciveis (Silver): categorias pai distintas
categorias_pereciveis = (
    spark.read.format("delta").options(**adls_options)
    .load(f"{SILVER_BASE}/physical_produtos_pereciveis")
    .select(col("categoria_pai").alias("categoria"))
    .distinct()
)
print(f"   Categorias perecíveis (Squad 3 Silver): {categorias_pereciveis.count()}")

# Unificar categorias válidas
categorias_ref = categorias_ecommerce.union(categorias_pereciveis).distinct()
CATEGORIAS_VALIDAS = [row.categoria for row in categorias_ref.collect()]
print(f"   Categorias válidas carregadas: {len(CATEGORIAS_VALIDAS)} (dinâmico)")

df = df.withColumn(
    "_flag_categoria_invalida",
    when(~col("categoria_fornecida").isin(CATEGORIAS_VALIDAS), lit(True))
    .otherwise(lit(False))
)

# Regra 4: tipo_fornecedor na lista permitida
TIPOS_PERMITIDOS = [
    "Produtor Rural", "Cooperativa Agrícola", "Indústria",
    "Indústria Frigorífica", "Produção Própria (Loja)",
    "Importador", "Distribuidor"
]
df = df.withColumn(
    "_flag_tipo_invalido",
    when(~col("tipo_fornecedor").isin(TIPOS_PERMITIDOS), lit(True))
    .otherwise(lit(False))
)

# Regra 5: lead_time_dias > 0 e range por tipo
df = df.withColumn(
    "_flag_lead_time_invalido",
    when(col("lead_time_dias").isNull() | (col("lead_time_dias") <= 0), lit(True))
    .when((col("tipo_fornecedor") == "Produção Própria (Loja)") &
          ((col("lead_time_dias") < 1) | (col("lead_time_dias") > 3)), lit(True))
    .when((col("tipo_fornecedor") != "Produção Própria (Loja)") &
          ((col("lead_time_dias") < 2) | (col("lead_time_dias") > 20)), lit(True))
    .otherwise(lit(False))
)

# Extra: UF válida do Brasil
UFS_BRASIL = [
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS",
    "MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC",
    "SP","SE","TO"
]
df = df.withColumn(
    "_flag_uf_invalida",
    when(~col("uf_origem").isin(UFS_BRASIL), lit(True)).otherwise(lit(False))
)

# Extra: razao_social não nula/vazia
df = df.withColumn(
    "_flag_razao_social_vazia",
    when((col("razao_social").isNull()) | (trim(col("razao_social")) == ""), lit(True))
    .otherwise(lit(False))
)

print("   Etapa 5 — Flags de qualidade aplicadas")

# ── ETAPA 6: AUDITORIA E PARTIÇÃO ───────────────────────────────────────
df = (
    df
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("ano", year("dt_cadastro"))
    .withColumn("mes", month("dt_cadastro"))
)

df_silver_fornecedores = df

# ── Resumo ────────────────────────────────────────────────────────────────
total = df_silver_fornecedores.count()
print(f"\n   ✅ Pronto para Silver: {total} registros")
for flag_col in [c for c in df_silver_fornecedores.columns if c.startswith("_flag_")]:
    flagged = df_silver_fornecedores.filter(col(flag_col) == True).count()
    pct = (flagged / total * 100) if total > 0 else 0
    s = "✅" if flagged == 0 else "⚠️"
    print(f"   {s} {flag_col}: {flagged}/{total} ({pct:.1f}%)")

## BLOCO 4 — Transformação Silver: `food_lotes_producao`

Mesmas 6 etapas aplicadas aos lotes. A validação de SKU (T2) e de fornecedor (T3)
depende das referências carregadas no Bloco 2 e do `df_silver_fornecedores`
produzido no Bloco 3.

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# SILVER — food_lotes_producao
# Etapas: Dedup → Remover colunas Bronze → Cast → Trim →
#         Validação (flags) → Auditoria
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🔧 Silver: food_lotes_producao")
print("=" * 60)

df = df_bronze_lotes

# ── ETAPA 1: DEDUPLICAÇÃO (Regra 1 — PK) ────────────────────────────────
antes = df.count()
df = df.filter(col("id_lote").isNotNull())
w = Window.partitionBy("id_lote").orderBy(col("bronze_ingested_at").desc())
df = df.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")
depois = df.count()
print(f"   Etapa 1 — Dedup PK: {antes} → {depois} (removidos {antes - depois})")

# ── ETAPA 2: REMOVER COLUNAS DE AUDITORIA DA BRONZE ─────────────────────
df = df.drop("bronze_ingested_at", "bronze_source_file", "ano", "mes")
print("   Etapa 2 — Colunas Bronze removidas")

# ── ETAPA 3: CAST EXPLÍCITO DE TIPOS ────────────────────────────────────
df = (
    df
    .withColumn("id_lote", col("id_lote").cast(IntegerType()))
    .withColumn("id_fornecedor", col("id_fornecedor").cast(IntegerType()))
    .withColumn("quantidade_produzida", col("quantidade_produzida").cast(IntegerType()))
    .withColumn("dt_fabricacao", col("dt_fabricacao").cast(TimestampType()))
    .withColumn("dt_validade", col("dt_validade").cast(TimestampType()))
)
print("   Etapa 3 — Cast de tipos aplicado")

# ── ETAPA 4: TRIM DE STRINGS ────────────────────────────────────────────
df = (
    df
    .withColumn("sku", trim(col("sku")))
    .withColumn("temperatura_armazenamento_ideal", trim(col("temperatura_armazenamento_ideal")))
    .withColumn("status", trim(col("status")))
)
print("   Etapa 4 — Trim aplicado")

# ── ETAPA 5: VALIDAÇÃO — FLAGS DE QUALIDADE ─────────────────────────────

# Regra 2: SKU deve existir no catálogo unificado
df = df.join(
    skus_validos, df["sku"] == skus_validos["sku_catalogo"], "left"
).withColumn(
    "_flag_sku_sem_catalogo",
    when(col("sku_catalogo").isNull(), lit(True)).otherwise(lit(False))
).drop("sku_catalogo")

# Regra 3: id_fornecedor deve existir em food_fornecedores Silver
fornecedores_ids = (
    df_silver_fornecedores
    .select(col("id_fornecedor").alias("id_forn_ref"))
    .distinct()
)
df = df.join(
    fornecedores_ids, df["id_fornecedor"] == fornecedores_ids["id_forn_ref"], "left"
).withColumn(
    "_flag_fornecedor_orfao",
    when(col("id_forn_ref").isNull(), lit(True)).otherwise(lit(False))
).drop("id_forn_ref")

# Regra 4: status no enum válido
STATUS_VALIDOS = ["Ativo", "Vencido", "Recall"]
df = df.withColumn(
    "_flag_status_invalido",
    when(~col("status").isin(STATUS_VALIDOS), lit(True)).otherwise(lit(False))
)

# Regra 5: dt_validade > dt_fabricacao
df = df.withColumn(
    "_flag_validade_invertida",
    when(col("dt_validade") <= col("dt_fabricacao"), lit(True)).otherwise(lit(False))
)

# Extra: temperatura_armazenamento_ideal no enum
TEMPERATURAS_VALIDAS = ["Refrigerado", "Ambiente", "Congelado"]
df = df.withColumn(
    "_flag_temperatura_invalida",
    when(~col("temperatura_armazenamento_ideal").isin(TEMPERATURAS_VALIDAS), lit(True))
    .otherwise(lit(False))
)

# Extra: quantidade_produzida > 0
df = df.withColumn(
    "_flag_quantidade_invalida",
    when((col("quantidade_produzida").isNull()) | (col("quantidade_produzida") <= 0), lit(True))
    .otherwise(lit(False))
)

print("   Etapa 5 — Flags de qualidade aplicadas")

# ── ETAPA 6: AUDITORIA E PARTIÇÃO ───────────────────────────────────────
df = (
    df
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("ano", year("dt_fabricacao"))
    .withColumn("mes", month("dt_fabricacao"))
)

df_silver_lotes = df

# ── Resumo ────────────────────────────────────────────────────────────────
total = df_silver_lotes.count()
print(f"\n   ✅ Pronto para Silver: {total} registros")
for flag_col in [c for c in df_silver_lotes.columns if c.startswith("_flag_")]:
    flagged = df_silver_lotes.filter(col(flag_col) == True).count()
    pct = (flagged / total * 100) if total > 0 else 0
    s = "✅" if flagged == 0 else "⚠️"
    print(f"   {s} {flag_col}: {flagged}/{total} ({pct:.1f}%)")

## BLOCO 5 — Função de gravação (merge manual compatível com Unity Catalog)

Merge via DataFrame — lê o Delta existente, une com o novo via
`unionByName(allowMissingColumns=True)`, deduplica por PK mantendo o mais recente
(`silver_processed_at.desc`) e sobrescreve. É equivalente a MERGE do ponto de
vista do resultado final, mas mais custoso em I/O. Aceitável porque Silver roda
semanalmente e as tabelas são pequenas (60 fornecedores, ~11k lotes).

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# GRAVAÇÃO SILVER
# Merge Manual via DataFrame (union + dedup por PK + overwrite)
# ══════════════════════════════════════════════════════════════════════════

def save_silver_merge(df_new, silver_path, pk_col, table_name):
    """
    Merge manual: lê o Delta existente, une com dados novos,
    deduplica pela PK (mantém o mais recente) e sobrescreve.
    """
    print(f"\n{'='*60}")
    print(f"💾 Gravação Silver: {table_name}")
    print(f"   Destino: {silver_path}")
    print(f"   PK: {pk_col}")
    print(f"{'='*60}")

    try:
        df_existing = (
            spark.read.format("delta")
            .options(**adls_options)
            .load(silver_path)
        )
        existing_count = df_existing.count()
        print(f"   📂 Delta existente: {existing_count} registros → MERGE")
        df_merged = df_existing.unionByName(df_new, allowMissingColumns=True)
    except Exception:
        print(f"   📂 Delta não existe → escrita inicial")
        df_merged = df_new

    w = Window.partitionBy(pk_col).orderBy(col("silver_processed_at").desc())
    df_final = (
        df_merged
        .withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1)
        .drop("_rn")
    )

    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ano", "mes")
        .options(**adls_options)
        .save(silver_path)
    )

    final_count = df_final.count()
    print(f"   ✅ Gravação concluída: {final_count} registros")

    df_check = spark.read.format("delta").options(**adls_options).load(silver_path)
    print(f"   ✅ Validação: {df_check.count()} registros confirmados")
    return df_check

print("✅ Função save_silver_merge definida.")

## BLOCO 6 — Gravação Silver

Executa a gravação das duas tabelas e valida pós-gravação lendo a Delta de volta.
A validação garante que o schema persistiu corretamente (relevante quando
`mergeSchema=true` está habilitado).

In [0]:
# ── Gravar food_fornecedores na Silver ───────────────────────────────────
df_silver_fornecedores_check = save_silver_merge(
    df_new=df_silver_fornecedores,
    silver_path=f"{SILVER_BASE}/food_fornecedores",
    pk_col="id_fornecedor",
    table_name="food_fornecedores"
)

df_silver_fornecedores_check.printSchema()
df_silver_fornecedores_check.show(5, truncate=False)

In [0]:
# ── Gravar food_lotes_producao na Silver ─────────────────────────────────
df_silver_lotes_check = save_silver_merge(
    df_new=df_silver_lotes,
    silver_path=f"{SILVER_BASE}/food_lotes_producao",
    pk_col="id_lote",
    table_name="food_lotes_producao"
)

df_silver_lotes_check.printSchema()
df_silver_lotes_check.show(5, truncate=False)

## BLOCO 7 — Relatório de qualidade

Varre todas as colunas `_flag_*` de ambas as tabelas e reporta a quantidade de
registros flagados por regra. Prova como as regras técnicas foram aplicadas e qual o impacto delas no dataset.

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# RELATÓRIO DE QUALIDADE — resumo final
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("📋 Relatório de Qualidade — Silver")
print("=" * 60)

for label, df_check in [("food_fornecedores", df_silver_fornecedores_check),
                         ("food_lotes_producao", df_silver_lotes_check)]:
    print(f"\n📦 {label}:")
    total = df_check.count()
    flag_cols = [c for c in df_check.columns if c.startswith("_flag_")]
    for flag_col in flag_cols:
        flagged = df_check.filter(col(flag_col) == True).count()
        pct = (flagged / total * 100) if total > 0 else 0
        s = "✅" if flagged == 0 else "⚠️"
        print(f"   {s} {flag_col}: {flagged}/{total} ({pct:.1f}%)")

print("\n" + "=" * 60)
print("🎉 Ingestão Silver concluída!")
print("   BRONZE (Delta) → SILVER (Delta + limpeza + tratamento + flags)")
print(f"\n   📁 food_fornecedores:   {df_silver_fornecedores_check.count()} registros")
print(f"   📁 food_lotes_producao: {df_silver_lotes_check.count()} registros")
print(f"\n   📍 Destino: {SILVER_BASE}/")
print("   🔄 Modo: merge (upsert por PK)")
print("   📊 Auditoria: silver_processed_at")
print("   📂 Partição: ano, mes")
print("\n   Tratamentos aplicados:")
print("   • Deduplicação por PK (registro mais recente)")
print("   • Cast explícito de tipos (INT, BOOLEAN, TIMESTAMP, STRING)")
print("   • Trim em todas as strings")
print("   • CNPJ normalizado (só dígitos, pontuação removida)")
print("   • Certificações normalizadas (trim por item separado por ;)")
print("   • UF padronizada (upper + trim)")
print("   • Colunas de auditoria Bronze removidas")
print("   • Validação de FK: SKU vs catálogo, id_fornecedor vs fornecedores")
print("   • Validação de enums: status, temperatura, tipo_fornecedor, UF")
print("   • Validação de ranges: lead_time, quantidade_produzida, validade")
print("=" * 60)